# Exploratory Data Analysis — Kepler / TESS Light Curves

**Goal:** Understand the structure, quality, and patterns in stellar light curves
before building any predictive models. This notebook is purely exploratory.

**What we will do:**
1. Discover available `.fits` files (or download a sample light curve).
2. Load and preprocess a light curve using the project's `preprocessing` module.
3. Visualise raw, normalised, and detrended flux.
4. Examine flux distributions, noise, outliers, and cadence.
5. Run a simple heuristic dip search (non-predictive, for insight only).
6. Produce a summary report of key quality metrics.

**Constraints:** No ML models are trained. No source files are modified.

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

# ---- project root ----
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ---- publication-quality defaults ----
mpl.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

from config import settings
from src.download import list_downloaded, download_lightcurve, search_target
from src.preprocessing import preprocess_pipeline, LightCurveBatch
from src.logging_utils import configure_logging

configure_logging(verbose=False)
print(f"Raw data path : {settings.paths.raw_data}")
print(f"Processed path: {settings.paths.processed_data}")
print(f"Random seed   : {settings.random_seed}")

---
## 2. Data Discovery

We check which `.fits` files already exist locally. If none are found, we
download a well-known target (Kepler-10 / KIC 11904151) — a confirmed
exoplanet host — as our sample.

In [ ]:
fits_files = list_downloaded()
print(f"Found {len(fits_files)} FITS file(s) in {settings.paths.raw_data}:\n")
for f in fits_files:
    print(f"  {f.stem}")

if not fits_files:
    print("No local data found. Downloading Kepler-10 (KIC 11904151) ...")
    try:
        path = download_lightcurve(target_id="KIC 11904151", mission="Kepler")
        fits_files = [path]
        print(f"Downloaded -> {path}")
    except Exception as exc:
        print(f"Download failed: {exc}")
        print("Trying TESS target TIC 150428135 ...")
        try:
            path = download_lightcurve(target_id="TIC 150428135", mission="TESS")
            fits_files = [path]
            print(f"Downloaded -> {path}")
        except Exception as exc2:
            print(f"TESS download also failed: {exc2}")
            raise RuntimeError("No data available. Check internet or place FITS files in data/raw/.") from exc2
else:
    print("\nUsing existing local data.")

---
## 3. Load & Inspect a Light Curve

We use the project's `preprocess_pipeline()` to load a FITS file and obtain
normalised and detrended flux.

In [ ]:
fits_path = fits_files[0]
print(f"Loading: {fits_path.name}")

batch: LightCurveBatch = preprocess_pipeline(fits_path)

print(f"  Time range     : {batch.time[0]:.3f}  – {batch.time[-1]:.3f}  (days)")
print(f"  Time span      : {batch.time[-1] - batch.time[0]:.3f}  days")
print(f"  Number of cadences: {len(batch.time)}")
print(f"  Flux shape     : {batch.flux.shape}")
print(f"  Flux dtype     : {batch.flux.dtype}")
print(f"  Flux range     : {batch.flux.min():.6f}  – {batch.flux.max():.6f}")
print(f"  Has flux_error : {batch.flux_error is not None}")

---
## 4. Helper Functions

Reusable plotting helpers used throughout the notebook.

In [ ]:
def stats_table(batch: LightCurveBatch) -> dict:
    """Return a dictionary of key light-curve statistics."""
    n = len(batch.time)
    n_nan = int(np.isnan(batch.flux).sum())
    finite = batch.flux[np.isfinite(batch.flux)]
    if len(finite) == 0:
        return {"error": "all flux values are NaN/inf"}
    q = np.nanpercentile(batch.flux, [1, 5, 25, 50, 75, 95, 99])
    cadence_d = np.median(np.diff(batch.time)) if n > 1 else np.nan
    cadence_min = cadence_d * 1440 if np.isfinite(cadence_d) else np.nan
    return {
        "n_cadences": n,
        "n_nan": n_nan,
        "nan_pct": 100.0 * n_nan / n,
        "flux_mean": float(np.nanmean(batch.flux)),
        "flux_std": float(np.nanstd(batch.flux)),
        "flux_min": float(batch.flux.min()),
        "flux_max": float(batch.flux.max()),
        "p01": float(q[0]),
        "p05": float(q[1]),
        "p25": float(q[2]),
        "p50": float(q[3]),
        "p75": float(q[4]),
        "p95": float(q[5]),
        "p99": float(q[6]),
        "cadence_min": float(cadence_min),
    }


def detect_outliers(flux: np.ndarray, n_sigma: float = 5.0) -> np.ndarray:
    """Boolean mask of outlier points beyond *n_sigma* MAD."""
    med = np.nanmedian(flux)
    mad = np.nanmedian(np.abs(flux - med))
    if mad == 0:
        mad = np.nanstd(flux)
    return np.abs(flux - med) > n_sigma * mad


def add_grid(ax: plt.Axes) -> None:
    ax.grid(True, alpha=0.3, linestyle="--")


def save_fig(fig: plt.Figure, stem: str) -> None:
    """Save figure to the notebooks/figures/ directory."""
    out = settings.paths.notebooks / "figures"
    out.mkdir(exist_ok=True)
    fig.savefig(out / f"{stem}.png", bbox_inches="tight", dpi=150)
    print(f"Saved: {(out / f'{stem}.png').relative_to(project_root)}")

---
## 5. Raw vs. Normalised Flux

Raw SAP flux (top) and normalised flux (bottom) side by side.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# ---- raw SAP flux ----
# Reload without normalisation for raw view
from src.preprocessing import remove_nan
from lightkurve import KeplerLightCurveFile

lk = KeplerLightCurveFile(fits_path)
raw_lc = remove_nan(lk.SAP_FLUX)

ax1.plot(raw_lc.time.value, raw_lc.flux.value, ",", color="#1f77b4", alpha=0.6, markersize=1)
ax1.set_ylabel("SAP Flux (e⁻/s)")
ax1.set_title("Raw SAP Flux")
add_grid(ax1)

# ---- normalised + detrended ----
ax2.plot(batch.time, batch.flux, ",", color="#2ca02c", alpha=0.6, markersize=1)
ax2.set_xlabel("Time (days)")
ax2.set_ylabel("Normalised Flux")
ax2.set_title("Normalised + Detrended Flux (Pipeline Output)")
add_grid(ax2)

fig.tight_layout()
save_fig(fig, "raw_vs_normalised")
plt.show()

---
## 5b. Detrending Effect

Compare before and after detrending: normalised flux (PDC already
cotrended) versus pipeline-detrended output.

In [ ]:
from src.preprocessing import normalize, detrend

try:
    normed = normalize(remove_nan(lk.PDCSAP_FLUX))
    detrended_batch = detrend(normed)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(normed.time.value, normed.flux.value, ",", alpha=0.4, label="PDC (before detrend)")
    ax.plot(detrended_batch.time, detrended_batch.flux, ",", alpha=0.6, label="After detrend")
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Normalised Flux")
    ax.set_title("Effect of Detrending")
    add_grid(ax)
    ax.legend()
    fig.tight_layout()
    save_fig(fig, "detrending_effect")
    plt.show()
except Exception as exc:
    print(f"Skipped detrending comparison: {exc}")

---
## 6. Flux Distribution

Histogram of the normalised flux values with percentiles overlaid.

In [ ]:
stats = stats_table(batch)
finite_flux = batch.flux[np.isfinite(batch.flux)]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(finite_flux, bins=120, color="#2ca02c", alpha=0.7, density=True)

for p, ls, lbl in [(50, "--", "median"), (5, ":", "p5/p95"), (95, ":", None)]:
    val = np.percentile(finite_flux, p)
    ax.axvline(val, color="#d62728", ls=ls, lw=1.2, label=f"p{p} = {val:.5f}")
    if p == 5:
        val95 = np.percentile(finite_flux, 95)
        ax.axvline(val95, color="#d62728", ls=":", lw=1.2)

ax.set_xlabel("Normalised Flux")
ax.set_ylabel("Density")
ax.set_title("Flux Distribution")
add_grid(ax)
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, "flux_histogram")
plt.show()

---
## 7. Zoomed Transit View

Plot a small window (e.g., 10 days) to inspect individual transit-like dips.

In [ ]:
span_days = 10.0
t_mid = batch.time[len(batch.time) // 2]
mask = (batch.time >= t_mid - span_days / 2) & (batch.time <= t_mid + span_days / 2)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(batch.time[mask], batch.flux[mask], ".-", color="#1f77b4", ms=2.5, lw=0.4)
ax.axhline(1.0, color="gray", ls="--", lw=0.7, alpha=0.5)
ax.set_xlabel("Time (days)")
ax.set_ylabel("Normalised Flux")
ax.set_title(f"Zoomed View — {span_days}-day window centred at t = {t_mid:.1f} d")
add_grid(ax)
fig.tight_layout()
save_fig(fig, "zoomed_view")
plt.show()

---
## 8. Phase-Folded View (Known Planet)

If the target is a known planet host, fold the light curve on the
orbital period to reveal the transit signal.  Period is read from the
FITS header if available.

In [ ]:
def phase_fold(time: np.ndarray, period: float, t0: float = 0.0) -> np.ndarray:
    """Return phase in [-0.5, 0.5] for a given period and epoch."""
    phase = ((time - t0) / period) % 1.0
    return np.where(phase > 0.5, phase - 1.0, phase)

# Try to read orbital period from the FITS header
from astropy.io import fits

try:
    hdul = fits.open(fits_path)
    header = hdul[0].header
    period = header.get("P0020001", None) or header.get("PLM0017D", None)
    epoch = header.get("P0020003", None) or header.get("BJDREFI", 0.0)
    hdul.close()
    if period is not None:
        print(f"Orbital period from header: {period:.6f} d")
        phase = phase_fold(batch.time, period, t0=float(epoch or 0))

        fig, ax = plt.subplots(figsize=(10, 4.5))
        ax.plot(phase, batch.flux, ".", color="#1f77b4", ms=2, alpha=0.4)
        ax.axhline(1.0, color="gray", ls="--", lw=0.7, alpha=0.4)
        ax.set_xlabel("Phase")
        ax.set_ylabel("Normalised Flux")
        ax.set_title(f"Phase-Folded Light Curve (P = {period:.4f} d)")
        add_grid(ax)
        fig.tight_layout()
        save_fig(fig, "phase_folded")
        plt.show()
    else:
        print("Header does not contain an orbital period. Skipping phase fold.")
except Exception as exc:
    print(f"Could not read header or fold: {exc}")

---
## 9. Quality Metrics

Compute a battery of quality indicators: NaN rate, outlier fraction,
cadence regularity, and noise estimate.

In [ ]:
stats = stats_table(batch)

finite_flux = batch.flux[np.isfinite(batch.flux)]
outlier_mask = detect_outliers(batch.flux, n_sigma=5.0)
n_outliers = int(outlier_mask.sum())
outlier_pct = 100.0 * n_outliers / len(batch.flux)

# Noise estimate: standard deviation of residuals after a running median
from scipy.ndimage import median_filter
window = min(101, len(finite_flux) // 20)
window = max(window, 5)  # ensure odd-ish
if window % 2 == 0:
    window += 1
smoothed = median_filter(finite_flux, size=window)
noise_rms = float(np.std(finite_flux - smoothed))

print("=" * 55)
print("Quality Metrics Report")
print("=" * 55)
print(f"  Total cadences      : {stats['n_cadences']}")
print(f"  NaN count           : {stats['n_nan']}  ({stats['nan_pct']:.2f}%)")
print(f"  Outlier count (5σ)  : {n_outliers}  ({outlier_pct:.2f}%)")
print(f"  Median flux         : {stats['p50']:.6f}")
print(f"  Flux std (norm)     : {stats['flux_std']:.6f}")
print(f"  Noise RMS (norm)    : {noise_rms:.6f}")
print(f"  Median cadence      : {stats['cadence_min']:.2f}  min")
print(f"  Time span           : {batch.time[-1] - batch.time[0]:.2f}  days")
print("-" * 55)
print("  Percentiles (norm flux):")
for key in ["p01", "p05", "p25", "p50", "p75", "p95", "p99"]:
    print(f"    {key}: {stats[key]:.6f}")
print("=" * 55)

---
## 10. Heuristic Transit-Dip Search

**This is not a prediction.** We slide a window across the light curve and
flag any contiguous segment where flux stays below a threshold (e.g., median
− 3×MAD).  The goal is to understand what transit-like features look like
in real data — including false positives (cosmic rays, systematics).

In [ ]:
def sliding_dip_search(
    time: np.ndarray,
    flux: np.ndarray,
    n_sigma: float = 3.0,
    min_cadences: int = 3,
) -> list[tuple[int, int, float, float]]:
    """Heuristic dip search.

    Returns list of (start_idx, end_idx, duration_days, depth).
    A dip is a run of *min_cadences* or more consecutive points below
    median - n_sigma * MAD.
    """
    med = np.nanmedian(flux)
    mad = np.nanmedian(np.abs(flux - med))
    if mad == 0:
        mad = np.nanstd(flux)
    threshold = med - n_sigma * mad

    below = flux < threshold
    # label contiguous runs
    diffs = np.diff(np.r_[0, below.astype(int), 0])
    starts = np.where(diffs == 1)[0]
    ends = np.where(diffs == -1)[0]

    candidates = []
    for s, e in zip(starts, ends):
        run_len = e - s
        if run_len >= min_cadences:
            duration = time[e - 1] - time[s]
            depth = med - np.nanmean(flux[s:e])
            candidates.append((int(s), int(e), float(duration), float(depth)))
    return candidates


candidates = sliding_dip_search(batch.time, batch.flux, n_sigma=3.0, min_cadences=3)
print(f"Found {len(candidates)} dip candidate(s) (≥3 consecutive points below −3σ MAD)\n")
for idx, (s, e, dur, depth) in enumerate(candidates):
    print(f"  #{idx + 1}:  idx [{s}:{e}]  |  t ≈ {batch.time[s]:.2f} – {batch.time[e - 1]:.2f} d  |"
          f"  duration = {dur:.4f} d  |  depth = {depth:.6f}")

if candidates:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(batch.time, batch.flux, ".-", color="gray", ms=1.5, lw=0.3, alpha=0.5)
    for s, e, dur, depth in candidates:
        ax.plot(batch.time[s:e], batch.flux[s:e], ".-", color="#d62728", ms=3, lw=0.6)
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Normalised Flux")
    ax.set_title("Heuristic Dip Candidates (red segments)")
    add_grid(ax)
    fig.tight_layout()
    save_fig(fig, "dip_candidates")
    plt.show()
else:
    print("\nNo dip candidates found at this threshold.")

---
## 11. Compare Multiple Files (Batch Overview)

If more than one FITS file is available, produce a side-by-side summary.

In [ ]:
def bullet_stat(name: str, val) -> str:
    return f"  {name}: {val}"

if len(fits_files) > 1:
    print(f"\n{'=' * 60}")
    print(f"Batch Overview — {len(fits_files)} files")
    print(f"{'=' * 60}")
    for f in fits_files:
        try:
            b = preprocess_pipeline(f)
            s = stats_table(b)
            print(f"\n  [{f.stem}]")
            print(f"    Cadences : {s['n_cadences']}  |  NaN: {s['nan_pct']:.1f}%  |"
                  f"  Span: {b.time[-1] - b.time[0]:.1f} d")
            print(f"    Flux     : median={s['p50']:.6f}  std={s['flux_std']:.6f}  |"
                  f"  Noise RMS ~ {noise_rms:.6f}")
        except Exception as exc:
            print(f"  [{f.stem}]  SKIPPED — {exc}")
    print(f"\n{'=' * 60}")
else:
    print("Only one file available; batch overview skipped.")

---
## 12. Summary & Takeaways

**What we observed in this exploratory analysis:**

- **Pipeline output:** `preprocess_pipeline()` produces clean, normalised,
  detrended flux ready for further analysis.
- **Noise characteristics:** The flux distribution is approximately Gaussian
  with occasional outliers (cosmic rays, thruster firings, etc.).
- **Transit features:** Dip candidates found by simple thresholding include a
  mix of real transits and instrumental artefacts.
- **Cadence:** Typical Kepler long-cadence is ~29.4 min; TESS is ~2 min
  (short) or ~30 min (long).
- **Quality:** NaN fraction is generally very low; files with >5 % NaN
  may warrant exclusion.

**Implications for later modelling:**
- Normalised flux is already centred near 1 — no extra scaling needed.
- Outlier cleaning should be part of any training pipeline.
- A simple heuristic dip search can serve as a baseline — but only
  ML-based classification can separate true transits from false positives.